<a href="https://colab.research.google.com/github/Dilukshika-Sasitharan/Statistical-Learning-e23355/blob/main/E23355_Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ASSIGNMENT#5 : INTRODUCTION TO DATA ANALYSIS**
# **E23355**
---

## Part A: Theoretical Fundamentals (Maximum Likelihood & Decision Space)

---

**Q1 · Bias of the MLE Covariance Estimator**

The MLE covariance estimator is:

$$\hat{\Sigma}_{MLE} = \frac{1}{n}\sum_{i=1}^{n}(x_i - \hat{\mu}_n)(x_i - \hat{\mu}_n)^T$$

Decompose deviation around the true mean μ:

$$x_i - \hat{\mu}_n = (x_i - \mu) - (\hat{\mu}_n - \mu)$$

Expand and use  $\sum_{i=1}^n(x_i - \mu) = n(\hat{\mu}_n - \mu)$:

$$\sum_{i=1}^n(x_i-\hat{\mu}_n)(x_i-\hat{\mu}_n)^T = \sum_{i=1}^n(x_i-\mu)(x_i-\mu)^T - n(\hat{\mu}_n-\mu)(\hat{\mu}_n-\mu)^T$$

Apply expectation using $E[(x_i-\mu)(x_i-\mu)^T]=\Sigma$ and $\text{Var}(\hat{\mu}_n)=\frac{1}{n}\Sigma$:

$$E[\hat{\Sigma}_{MLE}] = \frac{1}{n}(n\Sigma) - \frac{1}{n}(n \cdot \frac{1}{n}\Sigma) = \Sigma - \frac{1}{n}\Sigma = \frac{n-1}{n}\Sigma \neq \Sigma$$

The estimator is downward biased because deviations computed from the sample mean rather than the true mean artificially shrink estimated scatter.

---

**Bessel's Correction — deriving the unbiased estimator S:**

$$S = \frac{1}{n-1}\sum_{i=1}^n(x_i - \hat{\mu}_n)(x_i - \hat{\mu}_n)^T$$



$$E[S] = \frac{n}{n-1} \cdot \frac{n-1}{n}\Sigma = \Sigma \checkmark$$

S is the unique unbiased estimator of the population covariance matrix for any finite sample size n.

---

**Q2 · Type I and Type II Errors in Structural Health Monitoring**

| Error | Name | SHM Context |
|---|---|---|
| Type I (α) | False Alarm | Reject H₀ (flag fault) when structure is truly healthy → unnecessary plant shutdown |
| Type II (β) | Missed Detection | Fail to reject H₀ (no alarm) when genuine structural damage exists |

Consequence of α = 0.0001:

1. **Statistical power collapses:** Power = 1 − β drops sharply. Only extremely large anomalies cross the stringent threshold, so β (missed detections) rises dramatically — subtle damage goes undetected.

2. **Confidence ellipsoid expands:** The geometric volume of the healthy operation region scales with confidence level 1 − α = 0.9999. The ellipsoid swells to encompass almost all variation, making the system highly insensitive to genuine baseline degradation.

---

**Part B — Slutsky's Theorem & Asymptotic Distributions**

### Slutsky's Theorem (Rigorous Statement)

Let {Xₙ} and {Yₙ} be sequences of random vectors. If Xₙ →(d) X and Yₙ →(p) c (a constant), then:

$$X_n + Y_n \xrightarrow{d} X + c, \qquad Y_n X_n \xrightarrow{d} cX, \qquad X_n/Y_n \xrightarrow{d} X/c \;\;(c \neq 0)$$

---

### Justifying S → Σ substitution as n → ∞

Lindeberg-Lévy Multivariate CLT:

$$\sqrt{n}(\hat{\mu}_n - \mu) \xrightarrow{d} \mathcal{N}(0, \Sigma)$$

WLLN gives consistency of S:

$$S \xrightarrow{p} \Sigma \implies S^{-1/2} \xrightarrow{p} \Sigma^{-1/2}$$

Apply Slutsky — multiply distributional sequence by probability-limit matrix:

$$S^{-1/2} \cdot \sqrt{n}(\hat{\mu}_n - \mu) \xrightarrow{d} \Sigma^{-1/2} \cdot \mathcal{N}(0,\Sigma) = \mathcal{N}(0, I)$$

Rescale back to empirical metric space:

$$\hat{\mu}_n \sim \mathcal{N}\!\left(\hat{\mu}_n,\; \frac{1}{n}S\right)$$

Slutsky's Theorem mathematically justifies substituting the observable S for the latent Σ without corrupting the asymptotic distribution, operationalising the practical parametric monitoring envelope.


In [5]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# --- DO NOT MODIFY: Synthetic Strain Sensor Data Generator ---
np.random.seed(42)
n_samples = 5000
n_features = 3

base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01],
         [0.02, 0.06, 0.03],
         [0.01, 0.03, 0.05]],
    size=n_samples
)

# Injecting a subtle, hidden structural drift in the final 1,000 snapshots
base_data[4000:, 0] += 0.015   # Sensor 1 drift
base_data[4000:, 2] -= 0.010   # Sensor 3 drift

df_strain = pd.DataFrame(
    base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3']
)
# ----------------------------------------------------------------


def verify_first_moment_homogeneity(df: pd.DataFrame,
                                    g_chunks: int = 5) -> dict:
    """
    Partitions the dataset into g chunks and evaluates first-moment
    homogeneity via Wilks' Lambda and Bartlett's Chi-Square
    asymptotic transformation.

    Parameters
    ----------
    df       : DataFrame with multivariate observations (rows = snapshots)
    g_chunks : number of consecutive non-overlapping temporal blocks

    Returns
    -------
    dict with Wilks' Lambda, Chi2 statistic, df, p-value, conclusion
    """
    X = df.values
    n, m = X.shape
    g    = g_chunks
    chunk_size = n // g

    # 1. Split into g equal consecutive blocks
    chunks = [X[j * chunk_size : (j + 1) * chunk_size] for j in range(g)]

    # 2. Grand mean and per-chunk means
    grand_mean  = np.mean(X, axis=0)
    chunk_means = [np.mean(c, axis=0) for c in chunks]
    n_j         = chunk_size

    # 3. Within-chunk (W) and Between-chunk (B) scatter matrices
    W = np.zeros((m, m))
    for chunk in chunks:
        mu_j = np.mean(chunk, axis=0)
        dev  = chunk - mu_j
        W   += dev.T @ dev

    B = np.zeros((m, m))
    for mu_j in chunk_means:
        diff = (mu_j - grand_mean).reshape(-1, 1)
        B   += n_j * (diff @ diff.T)

    # 4. Wilks' Lambda:  Λ = |W| / |B + W|
    det_W    = np.linalg.det(W)
    det_BpW  = np.linalg.det(B + W)
    wilks_lambda = det_W / det_BpW

    # 5. Bartlett's Chi-Square approximation
    df_chi2   = m * (g - 1)
    chi2_stat = -(n - 1 - (m + g) / 2) * np.log(wilks_lambda)
    p_value   = 1 - stats.chi2.cdf(chi2_stat, df=df_chi2)

    conclusion = (
        "REJECT H0 — Baseline mean has shifted. First moment is NOT homogeneous."
        if p_value < 0.05 else
        "FAIL TO REJECT H0 — First moment is homogeneous across all chunks."
    )

    print("=" * 60)
    print("  MANOVA First-Moment Homogeneity Test")
    print("=" * 60)
    print(f"  Chunks (g)             : {g}")
    print(f"  Observations per chunk : {n_j}")
    print(f"  Feature dimensions (m) : {m}")
    print(f"  Wilks' Lambda (Λ)      : {wilks_lambda:.6f}")
    print(f"  Bartlett Chi² statistic: {chi2_stat:.4f}")
    print(f"  Degrees of freedom     : {df_chi2}")
    print(f"  p-value                : {p_value:.6f}")
    print("-" * 60)
    print(f"  Conclusion (α=0.05): {conclusion}")
    print("=" * 60)

    return dict(
        wilks_lambda       = wilks_lambda,
        chi2_statistic     = chi2_stat,
        degrees_of_freedom = df_chi2,
        p_value            = p_value,
        conclusion         = conclusion,
        W_matrix           = W,
        B_matrix           = B
    )


results = verify_first_moment_homogeneity(df_strain, g_chunks=5)

  MANOVA First-Moment Homogeneity Test
  Chunks (g)             : 5
  Observations per chunk : 1000
  Feature dimensions (m) : 3
  Wilks' Lambda (Λ)      : 0.990452
  Bartlett Chi² statistic: 47.9215
  Degrees of freedom     : 12
  p-value                : 0.000003
------------------------------------------------------------
  Conclusion (α=0.05): REJECT H0 — Baseline mean has shifted. First moment is NOT homogeneous.


# PCA Part A — Coordinate Projections & Orthogonality

**Q1 · Proving E[ZᵢZᵢᵀ] = Λ**

Expand using the projection Zᵢ = Pᵀ X̃ᵢ:

$$E[Z_i Z_i^T] = E[P^T \tilde{X}_i \tilde{X}_i^T P] = P^T \underbrace{E[\tilde{X}_i \tilde{X}_i^T]}_{=\,\Sigma} P = P^T \Sigma P$$

Substitute Σ = PΛPᵀ and use orthogonality PᵀP = I:

$$P^T(P\Lambda P^T)P = (P^T P)\,\Lambda\,(P^T P) = I \cdot \Lambda \cdot I = \Lambda \checkmark$$

**Geometric meaning:** Λ is strictly diagonal, so Cov(Zᵢ,ⱼ, Zᵢ,ₖ) = 0 for all j ≠ k.
The rotation Pᵀ maps the correlated raw sensor space into mutually uncorrelated principal
directions. Each axis independently captures a distinct, non-overlapping direction of variance.

---

**Q2 · Total variance conservation via the trace**

Using cyclic permutation invariance tr(ABC) = tr(CAB):

$$\text{tr}(\Sigma) = \text{tr}(P\Lambda P^T) = \text{tr}(P^T P\,\Lambda) = \text{tr}(I \cdot \Lambda) = \text{tr}(\Lambda) = \sum_{j=1}^m \lambda_j \checkmark$$

**Variance ratio formulas:**

$$\Phi(k) = \frac{\sum_{j=1}^k \hat{\lambda}_j}{\sum_{j=1}^m \hat{\lambda}_j} \qquad \text{(Cumulative Explained Variance Ratio)}$$

$$\Psi(k) = 1 - \Phi(k) = \frac{\sum_{j=k+1}^m \hat{\lambda}_j}{\sum_{j=1}^m \hat{\lambda}_j} \qquad \text{(Residual Unexplained Variance Ratio)}$$

---

**Q3 · Pythagorean decomposition and T² vs Q**

The centred observation partitions as:

$$(x_i - \hat{\mu}_n) = \hat{P}_k z_{i,k} + \hat{P}_{m-k} z_{i,m-k}$$

- **Reconstructed vector:** x̂ᵢ = P̂ₖ zᵢ,ₖ  (projection onto explained subspace)
- **Residual error vector:** eᵢ = P̂ₘ₋ₖ zᵢ,ₘ₋ₖ  (projection onto residual subspace)

Since P̂ₖ and P̂ₘ₋ₖ span orthogonal subspaces (P̂ₖᵀ P̂ₘ₋ₖ = 0):

$$\|x_i - \hat{\mu}_n\|^2 = \|z_{i,k}\|^2 + \|z_{i,m-k}\|^2 \checkmark$$

**T² vs Q diagnostic matching:**

| Statistic | Detects | Matched physical event |
|---|---|---|
| Hotelling's T² | Anomalies inside the principal subspace | Extreme wind loads — amplifies existing structural modes |
| Q (SPE) | Energy outside the subspace — new variance directions | Internal fatigue crack — alters load pathways outside baseline modes |



In [6]:
import numpy as np
import pandas as pd
from typing import Optional, Sequence, Dict, Any
from sklearn.decomposition import FactorAnalysis
import plotly.graph_objects as go
from plotly.subplots import make_subplots


class DualSubspaceDiagnosticsEngine:
    """
    Dual-mode diagnostic engine:
      - PCA: geometric de-correlation with 2x3 monitoring dashboard
      - FA : latent factor decomposition with 2x2 monitoring dashboard
    """

    def __init__(self, df: pd.DataFrame):
        self.df = df

    # ================================================================
    # PCA ENGINE — 2x3 Dashboard
    # ================================================================
    def compute_empirical_pca(
        self,
        columns: Optional[Sequence[str]] = None,
        show_plot: bool = True
    ) -> Dict[str, Any]:
        """
        Decomposes the unbiased empirical covariance matrix S into its
        orthogonal basis P_hat and renders a 2x3 diagnostic dashboard.
        """
        target_cols = (list(columns) if columns else
            [c for c in self.df.select_dtypes(include=[np.number]).columns
             if c != 'count'])

        X = self.df[target_cols].copy().dropna().values
        n, m = X.shape

        if n <= m:
            raise ValueError(f"Sample count n={n} must exceed feature count m={m}.")

        # 1. Centre
        mu_hat     = np.mean(X, axis=0)
        X_centered = X - mu_hat

        # 2. Unbiased sample covariance (Bessel ddof=1)
        S_matrix = np.cov(X, rowvar=False, ddof=1)

        # 3. Spectral decomposition, sorted descending
        eigenvalues, eigenvectors = np.linalg.eigh(S_matrix)
        idx        = np.argsort(eigenvalues)[::-1]
        lambda_hat = np.clip(eigenvalues[idx], 1e-15, None)
        P_hat      = eigenvectors[:, idx]

        # 4. Variance ratios
        total_variance            = np.sum(lambda_hat)
        explained_variance_ratio  = lambda_hat / total_variance
        cumulative_variance_ratio = np.cumsum(explained_variance_ratio)
        unexplained_variance_ratio = 1.0 - cumulative_variance_ratio

        # 5. Project to PC score space
        Z_scores = X_centered @ P_hat
        S_Z      = np.cov(Z_scores, rowvar=False, ddof=1)

        # 6. T² and Q across k in {1, ..., m-1}
        k_range      = np.arange(1, m)
        mean_T2_vs_k = []
        mean_Q_vs_k  = []
        T2_matrix    = np.zeros((n, len(k_range)))
        Q_matrix     = np.zeros((n, len(k_range)))

        for idx_k, k in enumerate(k_range):
            T2_samples = np.sum((Z_scores[:, :k] ** 2) / lambda_hat[:k], axis=1)
            T2_matrix[:, idx_k] = T2_samples
            mean_T2_vs_k.append(np.mean(T2_samples))

            Q_samples = np.sum(Z_scores[:, k:] ** 2, axis=1)
            Q_matrix[:, idx_k] = Q_samples
            mean_Q_vs_k.append(np.mean(Q_samples))

        print(f"\n-- PCA De-correlation Layer --")
        print(f"   Features: {m}  |  Samples: {n}")
        print(f"   Total System Variance tr(S): {total_variance:.4f}")
        for i, (ev, cv) in enumerate(zip(explained_variance_ratio,
                                         cumulative_variance_ratio)):
            print(f"   PC {i+1}: lambda={lambda_hat[i]:.4f}  "
                  f"Explained={ev*100:.2f}%  Cumulative={cv*100:.2f}%")

        # 7. 2x3 Dashboard
        if show_plot:
            pc_labels     = [f"PC {i+1}" for i in range(m)]
            k_labels      = [f"k={k}" for k in k_range]
            sensor_labels = target_cols

            fig = make_subplots(
                rows=2, cols=3,
                horizontal_spacing=0.18,
                vertical_spacing=0.28,
                subplot_titles=(
                    "Feature Loading Matrix |P_hat|",
                    "Component Eigenvalues (lambda)",
                    "Information Profile (Explained Var.%)",
                    "Residual Space (Unexplained Var.%)",
                    "Mean Hotelling's T^2 vs Subspace Size k",
                    "Mean Q Statistic (SPE) vs Subspace Size k"
                )
            )

            # R1C1: Loading Heatmap — left-anchored colorbar
            fig.add_trace(go.Heatmap(
                z=np.abs(P_hat), x=pc_labels, y=sensor_labels,
                colorscale='YlOrRd',
                colorbar=dict(
                    title="Loading", x=-0.12, len=0.38,
                    y=0.78, yanchor="middle", xanchor="right",
                    titleside="top"),
                showscale=True, showlegend=False),
                row=1, col=1)
            fig.update_xaxes(title_text="Principal Axes", row=1, col=1)

            # R1C2: Eigenvalues
            fig.add_trace(go.Bar(
                x=pc_labels, y=lambda_hat, name="Eigenvalue lambda_j",
                marker=dict(color='#1f77b4',
                            line=dict(color='black', width=0.5))),
                row=1, col=2)
            fig.update_yaxes(title_text="Variance Magnitude", row=1, col=2)

            # R1C3: Marginal + Cumulative variance
            fig.add_trace(go.Bar(
                x=pc_labels, y=explained_variance_ratio * 100,
                name="Marginal Explained %",
                marker=dict(color='#ff7f0e',
                            line=dict(color='black', width=0.5))),
                row=1, col=3)
            fig.add_trace(go.Scatter(
                x=pc_labels, y=cumulative_variance_ratio * 100,
                mode='lines+markers', name='Cumulative %',
                line=dict(color='#2ca02c', width=2, dash='dash')),
                row=1, col=3)
            fig.update_yaxes(title_text="Captured Variance (%)",
                             range=[-2, 105], row=1, col=3)

            # R2C1: Unexplained variance
            fig.add_trace(go.Bar(
                x=pc_labels, y=unexplained_variance_ratio * 100,
                name="Remaining Noise %",
                marker=dict(color='#2ca02c',
                            line=dict(color='black', width=0.5))),
                row=2, col=1)
            fig.update_yaxes(title_text="Excluded Info (%)",
                             range=[-2, 105], row=2, col=1)

            # R2C2: Hotelling's T²
            fig.add_trace(go.Scatter(
                x=k_labels, y=mean_T2_vs_k,
                mode='lines+markers', name='Mean T^2',
                line=dict(color='#9467bd', width=2)),
                row=2, col=2)
            fig.update_yaxes(title_text="Average T^2 Metric", row=2, col=2)
            fig.update_xaxes(title_text="Truncation Cutoff (k)", row=2, col=2)

            # R2C3: Q / SPE
            fig.add_trace(go.Scatter(
                x=k_labels, y=mean_Q_vs_k,
                mode='lines+markers', name='Mean Q (SPE)',
                line=dict(color='#e377c2', width=2)),
                row=2, col=3)
            fig.update_yaxes(title_text="Average Residual Energy", row=2, col=3)
            fig.update_xaxes(title_text="Truncation Cutoff (k)", row=2, col=3)

            fig.update_layout(
                title=dict(
                    text="PCA Optimization & Feature Loading Dashboard",
                    x=0.5, y=0.97, xanchor='center', yanchor='top'),
                template="plotly_white",
                showlegend=True,
                legend=dict(orientation="h", yanchor="bottom", y=1.02,
                            xanchor="center", x=0.5),
                margin=dict(t=150, b=60, l=140, r=80),
                height=750, width=1250)
            fig.show()

        return dict(
            mean_vector               = mu_hat,
            covariance_matrix_S       = S_matrix,
            eigenvalues_lambda        = lambda_hat,
            eigenvectors_P            = P_hat,
            explained_variance_ratio  = explained_variance_ratio,
            cumulative_variance_ratio = cumulative_variance_ratio,
            unexplained_variance_ratio= unexplained_variance_ratio,
            transformed_scores_Z      = Z_scores,
            score_covariance_diagonal = np.diag(S_Z),
            features                  = target_cols,
            k_values                  = k_range,
            T2_matrix_vs_k            = T2_matrix,
            Q_matrix_vs_k             = Q_matrix,
            mean_T2_profile           = np.array(mean_T2_vs_k),
            mean_Q_profile            = np.array(mean_Q_vs_k)
        )

    # ================================================================
    # FA ENGINE — 2x2 Dashboard
    # ================================================================
    def compute_empirical_fa(
        self,
        k: int,
        columns: Optional[Sequence[str]] = None,
        show_plot: bool = True
    ) -> Dict[str, Any]:
        """
        Factor Analysis latent subspace framework with Varimax rotation.
        Renders a clean 2x2 diagnostic dashboard.
        """
        target_cols = (list(columns) if columns else
            [c for c in self.df.select_dtypes(include=[np.number]).columns
             if c != 'count'])

        X = self.df[target_cols].copy().dropna().values
        n, m = X.shape

        # Dimensionality guardrails
        if n <= m:
            raise ValueError(f"Snapshot count n={n} must exceed feature count m={m}.")
        if k >= m:
            raise ValueError(
                f"Latent factor count k={k} must be strictly less than m={m}.")

        # 1. Z-Score normalisation with zero-variance guard
        mu_hat  = np.mean(X, axis=0)
        std_hat = np.std(X, axis=0, ddof=1)
        std_hat[std_hat == 0] = 1e-15
        Z        = (X - mu_hat) / std_hat
        R_matrix = np.corrcoef(X, rowvar=False)

        # 2. FA with Varimax rotation
        fa = FactorAnalysis(n_components=k, rotation='varimax', random_state=42)
        fa.fit(Z)
        lambda_matrix = fa.components_.T   # shape (m, k)
        uniqueness    = fa.noise_variance_  # phi^2

        # 3. Communality h² and Uniqueness phi²
        communality = np.sum(lambda_matrix ** 2, axis=1)

        # 4. Thomson's regression factor scores
        F_scores       = fa.transform(Z)
        factor_variances = np.var(F_scores, axis=0, ddof=1)

        print(f"\n-- FA Latent Subspace Layer ({k} factors) --")
        print(f"   Sensors: {m}  |  Samples: {n}")
        print(f"   Average System Communality : {np.mean(communality)*100:.2f}%")
        print(f"   Average System Uniqueness  : {np.mean(uniqueness)*100:.2f}%")
        for j, (h2, phi2) in enumerate(zip(communality, uniqueness)):
            print(f"   {target_cols[j]:12s}  h2={h2*100:.2f}%  phi2={phi2*100:.2f}%")

        # 5. 2x2 Dashboard
        if show_plot:
            sensor_labels = target_cols
            factor_labels = [f"Factor {j+1}" for j in range(k)]

            fig = make_subplots(
                rows=2, cols=2,
                horizontal_spacing=0.24,
                vertical_spacing=0.28,
                subplot_titles=(
                    "Structural Loadings Matrix |lambda_(j,r)|",
                    "Variance Partitioning (h^2 vs phi^2)",
                    "Sensor Uniqueness Noise Floor (phi^2)",
                    "Latent Factor Scores Empirical Variance"
                )
            )

            # (1,1) Heatmap — left-anchored colorbar at x=-0.15
            fig.add_trace(go.Heatmap(
                z=np.abs(lambda_matrix),
                x=factor_labels, y=sensor_labels,
                colorscale='YlOrRd',
                colorbar=dict(
                    title="Sensitivity",
                    x=-0.15, len=0.38, y=0.78,
                    yanchor="middle", xanchor="right"),
                showscale=True, showlegend=False),
                row=1, col=1)
            fig.update_xaxes(title_text="Latent Structures", row=1, col=1)

            # (1,2) Horizontal stacked bar: h² blue + phi² orange
            fig.add_trace(go.Bar(
                y=sensor_labels, x=communality * 100,
                name="Communality h^2 (Shared Structure)",
                orientation='h',
                marker=dict(color='#1f77b4')),
                row=1, col=2)
            fig.add_trace(go.Bar(
                y=sensor_labels, x=uniqueness * 100,
                name="Uniqueness phi^2 (Channel Noise)",
                orientation='h',
                marker=dict(color='#ff7f0e')),
                row=1, col=2)
            fig.update_layout(barmode='stack')
            fig.update_xaxes(title_text="Variance Allocation (%)",
                             range=[0, 100], row=1, col=2)

            # (2,1) Uniqueness line — dot-dashed red, tickangle=25
            fig.add_trace(go.Scatter(
                x=sensor_labels, y=uniqueness,
                mode='lines+markers', name='Uniqueness phi^2',
                line=dict(color='#d62728', width=2, dash='dashdot'),
                marker=dict(symbol='x', size=10)),
                row=2, col=1)
            fig.update_yaxes(range=[-0.05, 1.05], row=2, col=1)
            fig.update_xaxes(title_text="Monitored Channels",
                             tickangle=25, row=2, col=1)

            # (2,2) Factor empirical variance — green bars, black border
            fig.add_trace(go.Bar(
                x=factor_labels, y=factor_variances,
                name="Factor Empirical Variance",
                marker=dict(color='#2ca02c',
                            line=dict(color='black', width=0.5))),
                row=2, col=2)
            fig.update_yaxes(title_text="Variance Level", row=2, col=2)
            fig.update_xaxes(title_text="Latent Vectors", row=2, col=2)

            fig.update_layout(
                title=dict(
                    text="FA Latent Subspace Diagnostics Dashboard",
                    x=0.5, y=0.97, xanchor='center', yanchor='top'),
                template="plotly_white",
                showlegend=True,
                legend=dict(
                    orientation="h", yanchor="bottom", y=1.02,
                    xanchor="center", x=0.5,
                    traceorder="normal", itemwidth=40,
                    itemsizing="constant"),
                margin=dict(t=150, b=60, l=140, r=80),
                height=750, width=1250)
            fig.show()

        return dict(
            mean_vector_mu         = mu_hat,
            std_vector_D           = std_hat,
            correlation_matrix_R   = R_matrix,
            factor_loadings_lambda = lambda_matrix,
            uniqueness_psi         = uniqueness,
            communality_h2         = communality,
            latent_factor_scores_F = F_scores,
            factor_variances       = factor_variances,
            sensors                = target_cols
        )

# FA Part A — Generative Model & Communalities

**Q1 · Proving the Fundamental Equation R = ΛΛᵀ + Ψ**

Start from the generative model Zᵢ = ΛFᵢ + εᵢ:

$$R = E[Z_i Z_i^T] = E[(\Lambda F_i + \epsilon_i)(\Lambda F_i + \epsilon_i)^T]$$

Expand using independence E[εᵢFᵢᵀ] = 0:
$$= \Lambda E[F_i F_i^T]\Lambda^T + E[\epsilon_i\epsilon_i^T]$$

Substitute E[FᵢFᵢᵀ] = I and E[εᵢεᵢᵀ] = Ψ:
$$R = \Lambda I \Lambda^T + \Psi = \Lambda\Lambda^T + \Psi \checkmark$$

**j-th diagonal entry:**
$$R_{jj} = 1 = \underbrace{\sum_{r=1}^k \lambda_{j,r}^2}_{h_j^2} + \varphi_j^2$$

| Parameter | Definition | Physical meaning |
|---|---|---|
| Communality h²ⱼ | Σᵣ λ²ⱼ,ᵣ | Shared variance explained by latent factors. High h² → sensor reflects true structural process |
| Uniqueness φ²ⱼ | 1 − h²ⱼ | Private sensor noise — calibration error, electrical fault. High φ² → instrument anomaly |

---

**Q2 · Varimax Rotation vs Raw PCA Loadings**

**Contrast:** A raw PCA loading vector is a mathematical artefact of variance-maximisation
with non-zero entries across all sensors. A Varimax-rotated FA loading pushes entries toward
0 or 1 (simple structure) — clean groupings of sensors onto distinct factors.

**Why rotation preserves h²ⱼ and R ≈ ΛΛᵀ + Ψ:**

Any orthogonal rotation T (TᵀT = I) transforms Λ → ΛT. Then:
$$(ΛT)(ΛT)^T = Λ T T^T Λ^T = ΛΛ^T$$

The product ΛΛᵀ — and hence every communality h²ⱼ = ‖λⱼ‖² — is invariant under
orthogonal rotation. Only individual column directions change, not the total shared variance.

---

**Q3 · Thomson's Regression Method**

Construct joint vector Yᵢ = [Zᵢᵀ, Fᵢᵀ]ᵀ. Its joint covariance is:

$$\Sigma_{YY} = \begin{bmatrix} R & \Lambda \\ \Lambda^T & I_{k \times k} \end{bmatrix}$$

Verify off-diagonal block:
$$E[Z_i F_i^T] = E[(\Lambda F_i + \epsilon_i)F_i^T] = \Lambda I + 0 = \Lambda$$

Apply conditional mean formula E[X₂|x₁] = μ₂ + Σ₂₁Σ₁₁⁻¹(x₁ − μ₁) with zero means:

$$\hat{f}_i = \Lambda^T R^{-1} z_i$$

In matrix form across all n snapshots:

$$\boxed{F = Z R^{-1} \Lambda_{\text{rotated}}}$$

Implemented via fa.transform(Z) in scikit-learn.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ================================================================
# CUSTOM GRAPH COLORS
# ================================================================
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=[
    '#E63946',  # Red
    '#457B9D',  # Blue
    '#2A9D8F',  # Teal
    '#F4A261',  # Orange
    '#6A4C93',  # Purple
    '#FF006E',  # Pink
    '#118AB2',  # Sky Blue
    '#06D6A0'   # Green
])

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#F8F9FA'
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['font.size'] = 11

# ================================================================
# DATA GENERATOR & FULL PIPELINE EXECUTION
# ================================================================
np.random.seed(42)
n_samples = 2500

# True latent physical driving forces
f1 = np.random.normal(0, 1, n_samples)
f2 = np.random.normal(0, 1, n_samples)

# Observable sensor channels with targeted noise profiles
s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.30, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)

df_asset = pd.DataFrame(
    np.column_stack([s1, s2, s3, s4]),
    columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4']
)

print("Dataset shape:", df_asset.shape)

print("\nEmpirical variances:")
print(df_asset.var(ddof=1).round(4))

print("\nCorrelation matrix:")
print(df_asset.corr().round(4))

# ================================================================
# PCA ANALYSIS
# ================================================================
engine = DualSubspaceDiagnosticsEngine(df_asset)

pca_metrics = engine.compute_empirical_pca(
    show_plot=True
)

# ================================================================
# FACTOR ANALYSIS
# ================================================================
fa_metrics = engine.compute_empirical_fa(
    k=2,
    show_plot=True
)

Dataset shape: (2500, 4)

Empirical variances:
Sensor_1    0.7944
Sensor_2    0.7640
Sensor_3    0.8917
Sensor_4    2.0173
dtype: float64

Correlation matrix:
          Sensor_1  Sensor_2  Sensor_3  Sensor_4
Sensor_1    1.0000    0.8593    0.1955    0.0355
Sensor_2    0.8593    1.0000    0.2666    0.0395
Sensor_3    0.1955    0.2666    1.0000    0.0566
Sensor_4    0.0355    0.0395    0.0566    1.0000

-- PCA De-correlation Layer --
   Features: 4  |  Samples: 2500
   Total System Variance tr(S): 4.4674
   PC 1: lambda=2.0360  Explained=45.57%  Cumulative=45.57%
   PC 2: lambda=1.5424  Explained=34.53%  Cumulative=80.10%
   PC 3: lambda=0.7819  Explained=17.50%  Cumulative=97.60%
   PC 4: lambda=0.1071  Explained=2.40%  Cumulative=100.00%



-- FA Latent Subspace Layer (2 factors) --
   Sensors: 4  |  Samples: 2500
   Average System Communality : 50.66%
   Average System Uniqueness  : 49.35%
   Sensor_1      h2=86.46%  phi2=13.50%
   Sensor_2      h2=87.52%  phi2=12.44%
   Sensor_3      h2=27.79%  phi2=72.36%
   Sensor_4      h2=0.87%  phi2=99.09%


# **Subspace Diagnostics — Analysis Questions 1 to 4**

---

### Question 1 · The Total Variance Illusion

**Part 1 — Physical nature of Sensor_4 uniqueness φ² ≈ 98%:**

A uniqueness value approaching 100% means Sensor_4 shares virtually zero covariance with
the rest of the array. It is completely decoupled from the true physical processes f1 and f2
driving the structural asset. Its variance consists almost entirely of isolated white noise
from a severe electrical instrumentation fault (σ² ≈ 2.0). There is no structural
information content in Sensor_4's signal.

**Part 2 — How Sensor_4 contaminates the PCA engine:**

PCA maximises global total variance without distinguishing type. Because Sensor_4 carries
the largest raw variance (σ² ≈ 2.0, roughly twice any other channel), the first principal
axis PC 1 dedicates itself almost entirely to tracking this single noisy sensor. Its
eigenvalue is inflated to λ₁ ≈ 2.0. Sensors 1, 2, and 3 — carrying the true shared
structural signal — are sidelined to lower-ranked components with near-zero loading on PC 1.

**Part 3 — Engineering risk of relying only on PCA:**

A single blown or short-circuited instrument permanently hijacks the primary monitoring axis
(PC 1), inflating its eigenvalue and completely distorting the principal subspace geometry.
The Hotelling's T² control chart built on this corrupted subspace becomes useless for
detecting genuine systemic structural failures — precisely the most safety-critical scenario
the system was designed to catch.

---

### Question 2 · Decoupling Structural Loading via Varimax Rotation

**Part 1 — Varimax creates simple structure:**

PCA enforces a strict variance-maximisation hierarchy (λ₁ > λ₂ > ...), compelling each
axis to spread loadings across many sensors simultaneously. This produces mathematically
forced cross-loadings with no physical interpretation.

Varimax rotation relaxes this entirely. By maximising loading variance within each factor
column — driving entries toward 0 or 1 — it rotates factor axes until each sensor loads
dominantly on exactly one factor. The result matches the true data generation process:
Sensors 1 and 2 group onto Factor 1 (driven by f1), Sensor 3 loads exclusively onto
Factor 2 (driven by f2).

**Part 2 — Operational interpretability advantage:**

With raw PCA, a diagnostic alarm on PC 2 requires decoding a complex combination of three
sensors. With the rotated FA heatmap:
- Factor 1 alarm → investigate Sensor_1 / Sensor_2 structural pathway (primary mode f1)
- Factor 2 alarm → isolate the independent Sensor_3 process (secondary mode f2)

Root cause diagnosis becomes instantaneous rather than requiring matrix algebra in the field.

---

### Question 3 · Determining Subspace Truncation k Using T² and Q

**Part 1 — Exact behaviour of the Mean Q curve:**

- k=1 → k=2: Mean Q (SPE) plunges sharply (approximately 2.4 down to 0.9). PC 2 captures
  the true structured shared variance of Sensors 1, 2, and 3 — eliminating substantial
  structured residual energy.

- k=2 → k=3: The curve flattens into a distinct elbow, decreasing only marginally
  (approximately 0.9 down to 0.1). PC 3 contributes only random residual noise.

**Part 2 — Why the elbow identifies k=2 as the true physical dimensionality:**

The Q statistic directly measures energy in the unexplained residual subspace. A sharp drop
signals the new component captures genuine structured variance. A flat elbow signals no more
structured information remains — subsequent components only absorb random noise. Therefore
k=2 is the true intrinsic physical dimension: PC 1 tracks Sensor_4 fault noise, PC 2
captures the shared structural signal of Sensors 1, 2, and 3.

**Part 3 — Danger of choosing k=3:**

PC 3 captures leftover residual variance of Sensor_3 — random noise with no physical
meaning. Setting k=3 forces this noise floor into the clean model subspace, causing T²
control limits to trigger frequent false alarms on normal baseline variation. This overfits
the monitoring system to noise and severely degrades practical reliability.

---

### Question 4 · Operational Trade-offs in System Health Monitoring

| Strategy | Mechanism | Response to single sensor failure | Robustness |
|---|---|---|---|
| PCA (T² + Q) | Tracks global principal subspace + residual space | Failing sensor restructures principal axes — corrupts T² for ALL healthy channels | Low |
| FA (latent factor scores) | Explicitly separates shared variance (h²) from localised noise (φ²) via diagonal Ψ | Single failure inflates that sensor's φ²→1, absorbed privately — shared factors remain clean | High |

**Recommendation: The FA strategy is more robust.**

The uniqueness metric φ²ⱼ acts as a natural circuit breaker — any sensor-level fault inflates
uniqueness rather than contaminating the shared latent subspace. Factor 1 and Factor 2 scores
remain uncorrupted by a single-sensor failure.

This is directly validated by the data: Sensor_4's φ² ≈ 98% demonstrates the mechanism —
its massive instrumentation noise is safely isolated within its own uniqueness term, leaving
the Factor 1 / Factor 2 representation of Sensors 1, 2, and 3 completely undisturbed. In the
PCA framework, that same noise hijacked PC 1 entirely and made the monitoring system blind
to real structural events.